# Práctica 01 · MLOps: evidencia de un experimento

**Versión para completar.** Construye dos *runs* comparables en MLflow con parámetros, métricas y artefactos de gobernanza. Trabaja en Databricks Free Edition: el experimento queda asociado a este notebook.

> No uses el modelo ni el dataset para decisiones clínicas. No registres secretos, datos personales ni prompts sensibles.

## 0. Dependencias

Ejecuta esta celda sólo si tu runtime no dispone de MLflow 3.1 o superior. Después reinicia Python y continúa desde la siguiente celda.

In [ ]:
%pip install -q --upgrade "mlflow[databricks]>=3.1" scikit-learn pandas

## 1. Configuración

Define un alias no personal y una configuración inicial. El segundo *run* deberá cambiar una sola decisión de modelo.

In [ ]:
import mlflow
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

STUDENT_ALIAS = "TODO-alias-sin-datos-personales"
RUN_LABEL = "TODO-baseline-heart"
MODEL_CONFIG = {
    "n_estimators": 80,
    "max_depth": TODO,  # Elige una profundidad.
    "min_samples_leaf": TODO,  # Elige un mínimo de muestras por hoja.
    "random_state": 42,
}

mlflow.set_tracking_uri("databricks")

## 2. Datos y artefactos de gobernanza

Carga el dataset didáctico y crea dos diccionarios: `dataset_card` y `risk_register`. El registro debe tener al menos tres riesgos, cada uno con impacto, propietario y mitigación.

In [ ]:
# Esta ruta funciona al abrir el repositorio como Databricks Git Folder.
DATASET_PATH = "../../../data/raw/heart.csv"
data = pd.read_csv(DATASET_PATH)
features = data.drop(columns="target")
target = data["target"]
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.2, random_state=42, stratify=target
)
categorical_columns = features.select_dtypes(include=["object", "bool"]).columns.tolist()
numeric_columns = [column for column in features.columns if column not in categorical_columns]
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", Pipeline([("impute", SimpleImputer(strategy="median"))]), numeric_columns),
        ("categorical", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("one_hot", OneHotEncoder(handle_unknown="ignore"))]), categorical_columns),
    ]
)

dataset_card = {
    "source": "TODO: referencia data/raw/heart.csv",
    "purpose": "TODO: limita el uso a la práctica docente",
    "rows": TODO,  # Usa el número de filas de data.
    "known_limitations": ["TODO", "TODO"],
}
risk_register = {
    "version": "s01",
    "risks": [
        {"id": "R1", "risk": "TODO", "impact": "TODO", "owner": "TODO", "mitigation": "TODO"},
        {"id": "R2", "risk": "TODO", "impact": "TODO", "owner": "TODO", "mitigation": "TODO"},
        {"id": "R3", "risk": "TODO", "impact": "TODO", "owner": "TODO", "mitigation": "TODO"},
    ],
}
display(features.head())

## 3. Run de entrenamiento

Completa el bloque para registrar configuración, tarjeta de datos, riesgos, métricas de prueba y el identificador de experimento. No registres el modelo: eso se trabajará cuando exista un contrato de entrada/salida.

In [ ]:
mlflow.sklearn.autolog(log_models=False, silent=True)

with mlflow.start_run(run_name=f"{STUDENT_ALIAS}-{RUN_LABEL}") as run:
    # TODO: registra tags de curso, semana, alias y nivel de riesgo.
    # TODO: registra MODEL_CONFIG, dataset_card y risk_register como artefactos JSON.
    model = Pipeline(
        steps=[
            ("preprocess", preprocessor),
            ("model", RandomForestClassifier(**MODEL_CONFIG)),
        ]
    )
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]

    metrics = {
        "test_f1": TODO,
        "test_recall": TODO,
        "test_roc_auc": TODO,
    }
    # TODO: registra metrics y guarda run_id y experiment_id.

# TODO: recupera el experimento con mlflow.get_experiment(experiment_id).

## 4. Comparación

Repite el entrenamiento cambiando únicamente `max_depth` o `min_samples_leaf`. Usa `mlflow.search_runs()` para mostrar nombre del run, parámetros y métricas. Ordena por `test_f1` descendente.

**Pregunta:** ¿Qué evidencia adicional necesitarías para decidir que el mejor F1 es apto para producción?

In [ ]:
# TODO: busca y muestra los runs de este experimento.
# Pista: filter_string="tags.course.week = '01'".
raise NotImplementedError("Completa la comparación de runs.")

## Entregable

Completa la ficha de `../examples/s01_project_record.yaml` con tus dos runs, métricas, decisión provisional y riesgos. En **Experiments**, verifica que tus artefactos no contienen secretos ni datos personales.